# 3 Strict Geocoding Accuracy
Compare Final_Lat/Final_Long against Strict Geocoding coordinates using haversine distance.
Creates maps showing disagreements and distribution statistics.

Input: data/1_derived/5_geocode_truck_stops/6_geocoded_strict.csv
Output: output/2_analysis/figures/geocoding_accuracy/ (HTML maps)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from math import radians, cos, sin, asin, sqrt


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
IN_FILE = PROJECT_ROOT / 'data/1_derived/5_geocode_truck_stops/6_geocoded_strict.csv'
FIG_DIR = PROJECT_ROOT / 'output' / '2_analysis' / 'figures' / 'geocoding_accuracy'
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(IN_FILE, low_memory=False)
df = df[df['Address_Type'] == 'Proper']
print(f'Loaded {len(df):,} Proper addresses')

# Check available coordinate columns
lat_lng_cols = [c for c in df.columns if 'lat' in c.lower() or 'lng' in c.lower() or 'lon' in c.lower()]
print(f'Coordinate columns: {lat_lng_cols}')

In [ ]:
# Compare Final vs Strict Geocoding on a map (lines where > 2 miles apart)
columns_info = [
    ('Final_Lat', 'Final_Long', 'Final Coordinates'),
    ('Strict_Geocoding_lat', 'Strict_Geocoding_lng', 'Strict Geocoding')
]

def parse_coords(lat_str, lon_str):
    exclude = {'nan', '', 'none', 'removed'}
    lats = [float(x) for x in str(lat_str).split(';') if x.strip().lower() not in exclude]
    lons = [float(x) for x in str(lon_str).split(';') if x.strip().lower() not in exclude]
    return list(zip(lats, lons))

fig = go.Figure()
min_dist = 2  # miles

for idx, row in df.iterrows():
    all_coords, hover_texts = [], []
    for lc, lnc, lbl in columns_info:
        if lc in row and lnc in row:
            coords = parse_coords(row[lc], row[lnc])
            if coords:
                all_coords.extend(coords)
                hover_texts.extend([f'Row: {idx}<br>Source: {lbl}'] * len(coords))
    show = False
    if len(all_coords) > 1:
        for i in range(1, len(all_coords)):
            lat1, lon1 = all_coords[i - 1]
            lat2, lon2 = all_coords[i]
            R = 3958.8
            dlat = radians(lat2 - lat1)
            dlon = radians(lon2 - lon1)
            a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
            d = R * 2 * asin(sqrt(a))
            if d >= min_dist:
                show = True
                break
    if show and len(all_coords) > 1:
        lats, lons = zip(*all_coords)
        fig.add_trace(go.Scattermapbox(
            lat=lats, lon=lons, mode='markers+lines',
            marker=dict(size=8), line=dict(width=2),
            name=f'Row {idx}', text=hover_texts, hoverinfo='text+lat+lon'
        ))

fig.update_layout(
    mapbox_style='open-street-map', mapbox_zoom=4,
    mapbox_center={'lat': 39.5, 'lon': -98.35},
    margin={'r': 0, 't': 0, 'l': 0, 'b': 0}, showlegend=True,
    title='Accuracy Comparison: Final vs Strict Geocoding'
)
fig.show()
fig.write_html(str(FIG_DIR / 'final_vs_strict_geocoding.html'))
print(f'Saved: {FIG_DIR / "final_vs_strict_geocoding.html"}')

In [ ]:
# Calculate haversine distances
def haversine_distance(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return None
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 3958.8 * 2 * asin(sqrt(a))

df['Distance_Final_vs_Strict'] = df.apply(
    lambda r: haversine_distance(r['Final_Lat'], r['Final_Long'], r['Strict_Geocoding_lat'], r['Strict_Geocoding_lng']),
    axis=1
)
valid = df['Distance_Final_vs_Strict'].dropna()

print('Accuracy Statistics: Final vs Strict Geocoding')
print('=' * 60)
print(f'Total comparisons: {len(valid)}')
print(f'Mean distance   : {valid.mean():.2f} miles')
print(f'Median distance : {valid.median():.2f} miles')
print(f'Std deviation   : {valid.std():.2f} miles')
print()
print('Distance Distribution:')
for threshold in [0.1, 0.5, 1, 2]:
    n = (valid <= threshold).sum()
    print(f'  Within {threshold} mi: {n} ({n / len(valid) * 100:.1f}%)')
print(f'  Beyond 2 mi  : {(valid > 2).sum()} ({(valid > 2).mean() * 100:.1f}%)')

In [ ]:
# Distribution plots
fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Histogram - Full Range', 'Histogram - Zoomed (0-10 mi)',
                    'Box Plot', 'Cumulative Distribution'))

fig.add_trace(go.Histogram(x=valid, nbinsx=50, name='Full'), row=1, col=1)
fig.add_trace(go.Histogram(x=valid[valid <= 10], nbinsx=50, name='0-10 mi'), row=1, col=2)
fig.add_trace(go.Box(y=valid, name='Distance'), row=2, col=1)

sorted_d = np.sort(valid)
fig.add_trace(go.Scatter(x=sorted_d, y=np.arange(1, len(sorted_d)+1)/len(sorted_d),
                         mode='lines', name='CDF'), row=2, col=2)

fig.update_layout(height=800, title_text='Distribution: Final vs Strict Geocoding', showlegend=False)
fig.show()

# Percentiles
print('Percentiles:')
for p in [25, 50, 75, 90, 95, 99]:
    print(f'  {p}th: {valid.quantile(p/100):.2f} miles')

In [ ]:
# Outlier removal (IQR) analysis
Q1 = valid.quantile(0.25)
Q3 = valid.quantile(0.75)
IQR = Q3 - Q1
no_outliers = valid[(valid >= Q1 - 1.5 * IQR) & (valid <= Q3 + 1.5 * IQR)]

print(f'Outlier Analysis:')
print(f'  Original points    : {len(valid)}')
print(f'  After IQR removal  : {len(no_outliers)}')
print(f'  Outliers removed   : {len(valid) - len(no_outliers)} ({(len(valid)-len(no_outliers))/len(valid)*100:.1f}%)')
print(f'  Upper bound (Q3+1.5*IQR): {Q3 + 1.5 * IQR:.2f} miles')
print()
print(f'Stats without outliers:')
print(f'  Mean  : {no_outliers.mean():.4f} miles')
print(f'  Median: {no_outliers.median():.4f} miles')
print(f'  Std   : {no_outliers.std():.4f} miles')

In [ ]:
# High-accuracy drill-down (<= 1 mile)
within_1 = valid[valid <= 1]
print(f'High-Accuracy Analysis (<= 1 mile):')
print(f'  Points within 1 mi: {len(within_1)} ({len(within_1)/len(valid)*100:.1f}%)')
print(f'  Mean  : {within_1.mean():.4f} miles')
print(f'  Median: {within_1.median():.4f} miles')
print()
for t in [0.01, 0.05, 0.1, 0.25, 0.5]:
    n = (within_1 <= t).sum()
    print(f'  Within {t} mi: {n} ({n/len(within_1)*100:.1f}%)')